# Results Explorer

Ready-to-run notebook for loading FD, all neural models, summary tables, raw results, and saved plots from `comparisons/results`.

This notebook does **not** retrain anything. It loads the saved benchmark outputs and artifacts so you can inspect numbers, policies, weights, and plots interactively.

In [ ]:
from pathlib import Path
import sys
import math

ROOT = Path.cwd().resolve()
while not (ROOT / 'comparisons').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    import pandas as pd
except ImportError:
    pd = None

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

from comparisons.core.notebook_api import (
    load_summary_table,
    list_available_runs,
    load_fd_policy_bundle,
    load_nn_model_bundle,
)
from comparisons.core.io import load_run_result
from comparisons.core.config import BenchmarkConfig
from comparisons.core.evaluation import run_real_data_portfolio_comparison
from real_data_loader import load_portfolio


In [ ]:
RESULTS_DIR = Path('comparisons/results')
N_ASSETS = 5
SEED = 1
INITIAL_WEALTH = 1.0
TARGET_MULTIPLIER = 1.10
TARGET_WEALTH = INITIAL_WEALTH * TARGET_MULTIPLIER

NN_METHODS = [
    'nn_mlp_small',
    'nn_mlp_deep',
    'deep_bsde',
    'pinn',
    'actor_critic',
    'lstm',
    'transformer',
]

print('Results dir:', RESULTS_DIR.resolve())
print('Using n_assets =', N_ASSETS, 'seed =', SEED, 'initial wealth =', INITIAL_WEALTH)

## Load Summary Tables

In [ ]:
main_results = load_summary_table('main_results', results_dir=RESULTS_DIR)
aggregated_results = load_summary_table('aggregated_results', results_dir=RESULTS_DIR)
neural_results = load_summary_table('neural_family_results', results_dir=RESULTS_DIR)
fd_vs_neural = load_summary_table('fd_vs_neural_results', results_dir=RESULTS_DIR)
available_runs = list_available_runs(results_dir=RESULTS_DIR)

print('main_results rows:', len(main_results))
print('aggregated_results rows:', len(aggregated_results))
print('neural_family_results rows:', len(neural_results))
print('available raw runs:', len(available_runs))

if pd is not None:
    main_df = pd.DataFrame(main_results)
    agg_df = pd.DataFrame(aggregated_results)
    neural_df = pd.DataFrame(neural_results)
    fd_nn_df = pd.DataFrame(fd_vs_neural)
    display(agg_df.head())
else:
    display(Markdown('`pandas` not installed; showing first aggregated row as dict.'))
    print(aggregated_results[0])

## Load Market Data Snapshot

In [ ]:
mkt = load_portfolio(N_ASSETS)
print(mkt)
print('History length:', mkt.log_ret.shape[0])

## Load FD Bundle

In [ ]:
fd_bundle = load_fd_policy_bundle(n_assets=N_ASSETS, seed=SEED, initial_wealth=INITIAL_WEALTH, results_dir=RESULTS_DIR)
fd_result = fd_bundle['result']
fd_policy = fd_bundle['policy']
fd_artifact = fd_bundle['artifact']

print('FD method:', fd_result['method_name'])
print('FD weight-path shape:', fd_result['weight_path'].shape)
print('FD policy sample at w=1.0, tau=0.5:', fd_policy(1.0, 0.5))

## Load All Neural Model Bundles

In [ ]:
nn_bundles = {
    name: load_nn_model_bundle(name, n_assets=N_ASSETS, seed=SEED, initial_wealth=INITIAL_WEALTH, results_dir=RESULTS_DIR)
    for name in NN_METHODS
}

for name, bundle in nn_bundles.items():
    weights = bundle['weights_fn'](INITIAL_WEALTH, TARGET_WEALTH)
    print(f'{name:<14} weights shape={weights.shape} sample norm={np.linalg.norm(weights):.3f}')

## Compare Instantaneous Weights at a Chosen State

In [ ]:
wealth_probe = 1.0
history_probe = [0.96, 0.99, 1.01, 1.03]
step_idx_probe = len(history_probe) - 1

fd_weight = np.array([fd_policy(wealth_probe / TARGET_WEALTH, 0.5)])
rows = []
for name, bundle in nn_bundles.items():
    weights = bundle['weights_fn'](
        wealth_probe,
        TARGET_WEALTH,
        history=history_probe,
        step_idx=step_idx_probe,
        total_steps=252,
    )
    rows.append((name, weights))

print('FD proxy weight:', fd_weight)
for name, weights in rows:
    print(name, np.round(weights, 4))

## Load Raw Runs For FD + All Neural Methods

In [ ]:
raw_runs = {
    'fd_1d_proxy': load_run_result(RESULTS_DIR / 'raw' / f'fd_1d_proxy_n{N_ASSETS}_seed{SEED}_w{INITIAL_WEALTH:.2f}.npz')
}
for name in NN_METHODS:
    raw_runs[name] = load_run_result(RESULTS_DIR / 'raw' / f'{name}_n{N_ASSETS}_seed{SEED}_w{INITIAL_WEALTH:.2f}.npz')

for name, run in raw_runs.items():
    print(name, 'terminal wealth =', float(run['terminal_wealth'][0]), 'weight_path shape =', run['weight_path'].shape)

## Wealth Paths

In [ ]:
plt.figure(figsize=(11, 5))
for name, run in raw_runs.items():
    plt.plot(run['wealth_path'], linewidth=1.5, label=name)
plt.title(f'Wealth Paths for n={N_ASSETS}, seed={SEED}')
plt.xlabel('Time index')
plt.ylabel('Wealth')
plt.legend(ncol=2)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Terminal Wealth Bar Chart

In [ ]:
labels = list(raw_runs.keys())
terminal_values = [float(raw_runs[name]['terminal_wealth'][0]) for name in labels]

plt.figure(figsize=(11, 4))
plt.bar(labels, terminal_values)
plt.xticks(rotation=30, ha='right')
plt.ylabel('Terminal wealth')
plt.title(f'Terminal Wealth Comparison for n={N_ASSETS}, seed={SEED}')
plt.tight_layout()
plt.show()

## Gross Leverage Comparison

In [ ]:
plt.figure(figsize=(11, 5))
for name, run in raw_runs.items():
    plt.plot(run['gross_leverage_path'], linewidth=1.2, label=name)
plt.title(f'Gross Leverage Paths for n={N_ASSETS}, seed={SEED}')
plt.xlabel('Time index')
plt.ylabel('Gross leverage')
plt.legend(ncol=2)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Neural Family Table Filtered to Current Dimension

In [ ]:
if pd is not None:
    neural_dim = neural_df[neural_df['n_assets'].astype(int) == N_ASSETS].copy()
    cols = [
        'method_name', 'n_assets', 'n_runs', 'target_hit_rate', 'mean_terminal_wealth',
        'train_time_sec', 'mean_gross_leverage', 'max_drawdown'
    ]
    display(neural_dim[cols].sort_values(['method_name']))
else:
    neural_dim = [row for row in neural_results if int(row['n_assets']) == N_ASSETS]
    for row in neural_dim:
        print(row)

## Show Saved Plot Pack

In [ ]:
plot_files = [
    'fd_vs_neural_target_hit_rate.png',
    'fd_vs_neural_mean_terminal_wealth.png',
    'fd_vs_neural_runtime.png',
    'neural_risk_vs_performance.png',
]
for plot_name in plot_files:
    display(Markdown(f'### {plot_name}'))
    display(Image(filename=str(RESULTS_DIR / 'plots' / plot_name)))

## Next Analysis Ideas

- change `N_ASSETS`, `SEED`, and `INITIAL_WEALTH` and rerun the notebook
- inspect `raw_runs[name]['weight_path']` for detailed allocation dynamics
- compare `fd_1d_proxy` against `fd_merton_multi` by loading the raw `.npz` files directly
- use `nn_bundles[name]['model']` for deeper architecture-specific inspection